# 05 Error Analysis
This notebook examines saved Linear SVM test errors and does not recompute predictions.

## Why Error Analysis Matters
Aggregate scores do not reveal which classes, confusion directions, or high-confidence cases create practical risk.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
ERRORS = ROOT / 'outputs' / 'error_analysis'
RESULTS = ROOT / 'outputs' / 'results'

summary = json.loads((ERRORS / 'baseline_error_summary.json').read_text(encoding='utf-8'))
misclassified = pd.read_csv(ERRORS / 'baseline_linear_svm_test_misclassified.csv', encoding='utf-8')
confusion_pairs = pd.read_csv(ERRORS / 'baseline_linear_svm_test_confusion_pairs.csv', encoding='utf-8')
class_errors = pd.read_csv(ERRORS / 'baseline_linear_svm_test_class_errors.csv', encoding='utf-8')
high_confidence = pd.read_csv(ERRORS / 'baseline_linear_svm_test_high_confidence_wrong.csv', encoding='utf-8')
minority_errors = pd.read_csv(ERRORS / 'baseline_linear_svm_test_minority_class_errors.csv', encoding='utf-8')
comparison = pd.read_csv(RESULTS / 'model_comparison_leaderboard.csv', encoding='utf-8')

## Linear SVM Error Summary
The saved test analysis contains 11,414 errors from 77,695 examples, an error rate of 14.69%.

In [ ]:
svm_test = summary['models']['linear_svm']['test']
pd.DataFrame({
    'Metric': ['Total rows', 'Correct', 'Errors', 'Error rate'],
    'Value': [svm_test['total_rows'], svm_test['total_correct'], svm_test['total_errors'], f"{svm_test['error_rate']:.2%}"],
})

## Worst Class: Neutral
Neutral is the worst class by error rate: 182 of 214 test examples are wrong, or 85.05%.

In [ ]:
display(class_errors.style.format({'error_rate': '{:.2%}'}))
class_errors.set_index('true_label')['error_rate'].plot(kind='bar', figsize=(7, 4), color=['#c44e52', '#8172b2', '#55a868'])
plt.title('Linear SVM Error Rate by True Class')
plt.ylabel('Error rate')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Most Common Confusion Pair
Negative → Positive is the largest saved confusion pair with 5,599 cases, followed by Positive → Negative with 5,388.

In [ ]:
display(confusion_pairs.style.format({'share_of_errors': '{:.2%}'}))
pairs = confusion_pairs.assign(pair=confusion_pairs['true_label'] + ' → ' + confusion_pairs['predicted_label'])
pairs.set_index('pair')['count'].sort_values().plot(kind='barh', figsize=(8, 4), color='#4c72b0')
plt.title('Linear SVM Test Confusion Pairs')
plt.xlabel('Errors')
plt.tight_layout()
plt.show()

## Positive/Negative Polarity Swaps
The two main polarity-swap directions account for 96.26% of all Linear SVM test errors, indicating difficulty with negation, sarcasm, context, and weak-label noise.

## High-Confidence Errors
High-confidence wrong predictions are especially important because the model appears decisive even when labels disagree.

In [ ]:
print(f"High-confidence wrong rows: {len(high_confidence):,}")
pd.set_option('display.max_colwidth', 100)
high_confidence[['clean_text', 'true_label', 'predicted_label', 'confidence', 'error_category']].head(10)

## Minority Class Errors
Neutral errors should be reviewed separately because very small support makes aggregate accuracy insensitive to this class.

In [ ]:
print(f"Saved Neutral-class errors: {len(minority_errors):,}")
minority_errors[['clean_text', 'true_label', 'predicted_label', 'confidence', 'error_category']].head(10)

## Sample Misclassified Tweets
These rows are sampled from the saved prediction output and retain both cleaned text and error categories for qualitative review.

In [ ]:
misclassified[['clean_text', 'true_label', 'predicted_label', 'confidence', 'text_length', 'error_category']].sample(10, random_state=42)

## Ethical Implications
The model should not be treated as ground truth for individual-level decisions. Weak labels, minority-class underperformance, and confident errors create fairness and misuse risks.

## Final Lessons Learned
Linear SVM is the strongest saved model, but 0.5040 Macro-F1 and 0.1303 Neutral F1 show substantial unresolved limitations. Reporting class-level errors and qualitative examples is necessary for an honest final evaluation.